# LLM Applications: Chat, Structured Output, Tool Calling, RAG

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/04-LLM-Apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers bitsandbytes sentence-transformers
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_json.py', 'q_tool.py', 'q_rag.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

## Load the model

Qwen3-1.7B takes about 3.5GB of memory in fp16.

In [ ]:
model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

In [ ]:
def chat(messages, max_new_tokens=512, thinking=False, **kwargs):
    """Take a list of messages and return the model's reply (a string)."""
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=thinking,
                                           return_dict=True, return_tensors="pt", **kwargs).to(model.device)
    gen_kwargs = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20) if thinking else \
                 dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(chat([{"role": "user", "content": "Explain the differences between PyTorch and NumPy in three points."}]))

## Multi-turn chat

The model itself has no memory. Its "memory" is simply the whole conversation history sent back in on every turn.

In [ ]:
history = [{"role": "system", "content": "You are a friendly data science teaching assistant. Keep your answers short."}]
for question in ["I want to learn deep learning. What should I learn first?", "How much math background do I need?", "What did you just suggest I learn first?"]:
    history.append({"role": "user", "content": question})
    answer = chat(history)
    history.append({"role": "assistant", "content": answer})
    print(f"👤 {question}\n🤖 {answer}\n")

## Batch generation

Processing several requests at once uses the GPU much more efficiently. Generative models need **left** padding (new tokens are appended on the right).

In [ ]:
tokenizer.padding_side = "left"
questions = ["What is the capital of France?", "What is 1+1?", "Please recommend a science fiction novel.", "Who created Python?"]
texts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False) for q in questions]
inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
for q, o in zip(questions, out):
    print(q, "→", tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()[:100])

## Structured output

For a program to use an LLM's output, you usually need a fixed format such as JSON.

In [ ]:
review = "Last weekend I tried that new ramen shop in the Xinyi district. The broth was rich and the chashu was tender, but we waited in line for an hour and a half, and NT$320 a bowl is a bit pricey."
prompt = f"""Extract information from the review below. Output only JSON, in this format:
{{"shop_type": str, "location": str, "price": int, "pros": [str], "cons": [str], "rating(1-5)": int}}

Review: {review}"""
answer = chat([{"role": "user", "content": prompt}])
print(answer)

In [ ]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)    # the model sometimes wraps it in ```json ... ``` or adds other text
    return json.loads(m.group(0))
extract_json(answer)

## Q
Write a function `parse_course(text)` that extracts `course_name`, `instructor`, `time` and `credits` from a course description and returns them as a dict.
Test it on several descriptions written in different styles to see whether it is reliable. If parsing fails, how could you handle it?

In [ ]:
# %load q_json.py
%run -i q_json.py

## Tool / function calling

LLMs are bad at arithmetic and don't know today's weather. But we can provide "tools": the model decides when to call one and with which arguments, our code runs it, and the result is sent back to the model. This is the foundation of **AI agents**.

`apply_chat_template(tools=...)` automatically builds the tool description (JSON schema) from the Python function's type hints and docstring.

In [ ]:
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.

    Args:
        city: The city name, e.g. "Taipei"
    """
    fake_data = {"Taipei": "Cloudy, 24°C", "Taichung": "Sunny, 28°C", "Kaohsiung": "Sunny, 30°C"}
    return fake_data.get(city, "No data found")

def calculator(expression: str) -> str:
    """
    Evaluate a math expression, e.g. "(3+5)*2".

    Args:
        expression: A math expression in Python syntax
    """
    return str(eval(expression, {"__builtins__": {}}))

tools = [get_weather, calculator]
print(tokenizer.apply_chat_template([{"role": "user", "content": "hi"}], tools=tools, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False))

In [ ]:
def parse_tool_calls(text):
    """Qwen3's tool-call format: <tool_call>{"name": ..., "arguments": {...}}</tool_call>"""
    return [json.loads(m) for m in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)]

def run_agent(question, max_turns=5):
    messages = [{"role": "user", "content": question}]
    available = {f.__name__: f for f in tools}
    for _ in range(max_turns):
        inputs = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, enable_thinking=False,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        calls = parse_tool_calls(text)
        if not calls:
            return text
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            result = available[c["name"]](**c["arguments"])
            print(f"🔧 {c['name']}({c['arguments']}) -> {result}")
            messages.append({"role": "tool", "name": c["name"], "content": result})
    return text

run_agent("What's the weather like in Taipei and Kaohsiung right now?")

In [ ]:
run_agent("What is 12345 times 6789, minus 1000?")

## Q
Add a tool `get_time()` that returns the current date and time (`datetime.now()`), then ask the model "What day of the week is it today?".

Note: small models don't always call the tool as they should; sometimes they just make up an answer (hallucination). Try different phrasings, or explicitly ask it to use a tool in the question, and compare the results.

In [ ]:
# %load q_tool.py
%run -i q_tool.py

## RAG (Retrieval-Augmented Generation)

An LLM doesn't know data it has never seen (for example your school's rules or a company's internal documents). RAG works like this:
1. Split the documents into small chunks and turn them into vectors with an **embedding model**
2. When the user asks a question, find the chunks most similar to the question
3. Put those chunks into the prompt together with the question

Here we use [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) with `sentence-transformers`.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
embedder.half()
docs = [
    "This course has three homework assignments, each worth 20% of the semester grade; the final project is worth 40%.",
    "Late homework loses 10% per day; homework more than one week late receives no credit.",
    "Final projects are done in groups of 2 to 3 and presented in week 16, with 10 minutes per group.",
    "TA office hours are Wednesdays from 2 to 4 PM in Room 305 of the Computer Science Building.",
    "This course uses the T4 GPU on Google Colab, so you don't need your own graphics card.",
    "Lectures are on Tuesdays from 9 AM to 12 PM in Room 201 of the General Classroom Building.",
]
doc_emb = embedder.encode(docs)
doc_emb.shape

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query], prompt_name="query")   # Qwen3-Embedding queries need an instruction prompt
    scores = embedder.similarity(q_emb, doc_emb)[0]
    top = scores.argsort(descending=True)[:k]
    return [(docs[i], scores[i].item()) for i in top]
retrieve("What happens if I hand in homework late?")

In [ ]:
def rag(question):
    context = "\n".join(f"- {d}" for d, _ in retrieve(question, k=3))
    prompt = f"Answer the question based on the information below. If the answer is not in the information, say you don't know.\n\nInformation:\n{context}\n\nQuestion: {question}"
    return chat([{"role": "user", "content": prompt}])
for q in ["What happens if my homework is three days late?", "Where can I go to ask the TA questions?", "What does the final exam cover?"]:
    print(f"👤 {q}\n🤖 {rag(q)}\n")

## Q
Pick a document of your own (for example a Wikipedia article or a course syllabus), split it into paragraphs, build a RAG system on it, and ask it a few questions.

Hint: `text.split("\n\n")` is a simple way to split paragraphs.

In [ ]:
# %load q_rag.py
%run -i q_rag.py

## Quantization: running larger models on a T4

Compressing the weights from 16 bits to 4 bits needs only about 1/4 of the memory, with little loss in quality.
Here we load Qwen3-4B in `bitsandbytes`' NF4 format (8GB in fp16, only about 2.5GB in 4-bit).

In [ ]:
import gc
del model, embedder
gc.collect(); torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
print(f"Memory: {model.get_memory_footprint()/1e9:.2f} GB")
print(chat([{"role": "user", "content": "Write a PyTorch function that computes the cosine similarity of two vectors."}]))